# STRATA slice viewer (v3 — Colab)

Run the cells top to bottom.

In [ ]:
#@title 📥 Step 1 — Download data (click ▶) { display-mode: "form" }
# ── Download data from Google Drive ──
!pip install -q gdown
import gdown, glob, os

FOLDER_ID = "1rSePBwwgEaTwP2JmWi4SxMeMcYYjY8M6"
BASE_DIR  = "/content/strata_forced_data/"
gdown.download_folder(id=FOLDER_ID, output=BASE_DIR, quiet=False)

DATA_DIR = os.path.dirname(glob.glob(BASE_DIR + "/**/labels_*.npy", recursive=True)[0]) + "/"

In [ ]:
#@title 📊 Step 2 — Launch viewer (click ▶) { display-mode: "form" }
%matplotlib inline
import numpy as np, matplotlib.pyplot as plt, ipywidgets as widgets, glob, re, math
from IPython.display import display
from matplotlib.colors import ListedColormap

DS  = 8
tag = f"ds{DS}"
_NxD, _NyD, _NzD = 4096 // DS, 2048 // DS, 4096 // DS

# Display labels swap simulation y <-> z (sim-y is vertical; shown as "z" here).
# "dim" is the numpy axis sliced — the single source of truth for slicing,
# slider range, and labels.
_AXIS = {
    "z": dict(dim=1, xl="$x$", yl="$y$", T=True),
    "y": dict(dim=2, xl="$x$", yl="$z$", T=True),
    "x": dict(dim=0, xl="$y$", yl="$z$", T=False),
}

# available (sigma, k) clustering combinations
_combos = sorted({(int(a), int(b)) for a, b in
    (re.search(r'sigma(\d+)_k(\d+)', f).groups()
     for f in glob.glob(DATA_DIR + f"labels_{tag}_sigma*_k*.npy"))})
SIGMA, K = _combos[0]

# load fields
_drdy = np.load(DATA_DIR + f"drdy_{tag}_norm.npy")
_leps = np.load(DATA_DIR + f"log_eps_{tag}.npy")
_lchi = np.load(DATA_DIR + f"log_chi_{tag}.npy")
_r    = np.load(DATA_DIR + f"r_{tag}.npy")

_bf_cache, _lbl_cache = {}, {}
def _load_clustering(sigma, k):
    if sigma not in _bf_cache:
        _bf_cache[sigma] = np.load(DATA_DIR + f"binary_filtered_{tag}_sigma{sigma}.npy")
    if (sigma, k) not in _lbl_cache:
        _lbl_cache[(sigma, k)] = np.load(DATA_DIR + f"labels_{tag}_sigma{sigma}_k{k}.npy")
    return _bf_cache[sigma], _lbl_cache[(sigma, k)]
_bf0, _lbl0 = _load_clustering(SIGMA, K)

_v = lambda a, p: float(np.nanpercentile(a[:, :, _NzD // 2], p))

def _drdy_slicer(axis, idx):
    # drdy_*_norm.npy is already normalized — return the slice as-is
    return np.take(_drdy, idx, axis=_AXIS[axis]["dim"])

FIELDS = [
    dict(data=_r,    title=r"Turbulent Density $\rho$", cmap="RdBu_r", vmin=-10, vmax=10),
    dict(slicer=_drdy_slicer, shape=(_NxD,_NyD,_NzD), title=r"Vertical density gradient $\partial \rho/\partial z$", cmap="RdBu_r", vmin=-10, vmax=10),
    dict(data=_bf0,  title=f"Filtered overturning fraction (σ={SIGMA/DS:.4g})", cmap="RdBu_r", vmin=0, vmax=0.5),
    dict(data=_lbl0, title=f"K-means clustering (k={K})"),
    dict(data=_leps, title=r"$\log_{10}(\varepsilon)$", cmap="inferno", vmin=_v(_leps,2), vmax=_v(_leps,98)),
    dict(data=_lchi, title=r"$\log_{10}(\chi)$",        cmap="inferno", vmin=_v(_lchi,2), vmax=_v(_lchi,98)),
]

_OVERLAY_SRC, _SIGMA_PANEL = 3, 2
_OVERLAY_COLORS = {4: "white", 5: "white"}   # ε / χ panels get white borders, rest black
_state = dict(axis="z", overlay=False, sigma=SIGMA, k=K)

def _slice(f, axis, idx):
    if "slicer" in f: return f["slicer"](axis, idx)
    return np.take(f["data"], idx, axis=_AXIS[axis]["dim"])

def _shape(f):
    return f["shape"] if "slicer" in f else f["data"].shape

_PANEL_W, _N_COLS = 4.5, 3
_N_ROWS = math.ceil(len(FIELDS) / _N_COLS)

def _make_figure(axis, idx):
    cfg, do_T = _AXIS[axis], _AXIS[axis]["T"]
    h0, w0 = _slice(FIELDS[0], axis, idx).shape
    panel_h = _PANEL_W * (w0 / h0) if do_T else _PANEL_W * (h0 / w0)
    fig, axs2d = plt.subplots(_N_ROWS, _N_COLS,
                              figsize=(_PANEL_W * _N_COLS, panel_h * _N_ROWS + 0.9), squeeze=False)
    plt.subplots_adjust(left=0.05, right=0.97, bottom=0.08, top=0.90, wspace=0.25, hspace=0.15)
    axs = [axs2d[r][c] for r in range(_N_ROWS) for c in range(_N_COLS)]
    for ax in axs[len(FIELDS):]: ax.set_visible(False)
    axs = axs[:len(FIELDS)]

    ext = None
    for i, (ax, f) in enumerate(zip(axs, FIELDS)):
        sl = _slice(f, axis, idx); h, w = sl.shape
        if do_T: data, ext = np.ma.masked_invalid(sl.T), [-0.5, h-0.5, -0.5, w-0.5]
        else:    data, ext = np.ma.masked_invalid(sl),   [-0.5, w-0.5, -0.5, h-0.5]
        if i == _OVERLAY_SRC:
            n = int(f["data"].max()) + 1
            cmap, vmin, vmax, ticks = ListedColormap(plt.get_cmap("tab10").colors[:n]), -0.5, n-0.5, list(range(n))
        else:
            cmap, vmin, vmax, ticks = f.get("cmap", "viridis"), f.get("vmin"), f.get("vmax"), None
        im = ax.imshow(data, origin="lower", aspect="equal", cmap=cmap,
                       interpolation="nearest", extent=ext, vmin=vmin, vmax=vmax)
        plt.colorbar(im, ax=ax, shrink=0.8, ticks=ticks)
        ax.set_title(f["title"], fontsize=9); ax.set_xlabel(cfg["xl"]); ax.set_ylabel(cfg["yl"])

    sd = _state["sigma"] / DS; pad = sd * 0.4
    axs[_SIGMA_PANEL].add_patch(plt.Circle((sd+pad, sd+pad), sd, fill=False, edgecolor="green", linewidth=3, zorder=5))
    axs[_SIGMA_PANEL].text(sd+pad, sd+pad, r"$\sigma$", color="green", ha="center", va="center", fontsize=7, zorder=6)

    if _state["overlay"]:
        lbl = _slice(FIELDS[_OVERLAY_SRC], axis, idx)
        kk = int(FIELDS[_OVERLAY_SRC]["data"].max()) + 1
        ld = lbl.T if do_T else lbl
        for i, ax in enumerate(axs):
            ax.contour(ld, levels=np.arange(0.5, kk), colors=_OVERLAY_COLORS.get(i, "black"),
                       linewidths=0.6, alpha=0.7, extent=ext)

    fig.suptitle(f"{axis}-slice {idx}  (σ={_state['sigma']}, k={_state['k']}, every {DS} grid pts)", fontsize=11)
    return fig

# ── interactive controls ──
axis_dd  = widgets.Dropdown(options=["z", "y", "x"], value="z", description="Slice axis:", style={"description_width": "initial"})
slider   = widgets.IntSlider(value=0, min=0, max=_shape(FIELDS[0])[_AXIS[_state["axis"]]["dim"]]-1, description="Slice index:",
                             continuous_update=False, style={"description_width": "initial"}, layout=widgets.Layout(width="500px"))
overlay  = widgets.Checkbox(value=False, description="Show cluster borders", style={"description_width": "initial"})
combo_dd = widgets.Dropdown(options=[(f"σ={s}, k={k}", (s, k)) for s, k in _combos], value=_combos[0],
                            description="Clustering:", style={"description_width": "initial"})
out = widgets.Output()
_busy = {"v": False}

def _refresh():
    with out:
        out.clear_output(wait=True)
        fig = _make_figure(_state["axis"], slider.value)
        plt.show(); plt.close(fig)

def _on_axis(c):
    _state["axis"] = c["new"]
    n = _shape(FIELDS[0])[_AXIS[c["new"]]["dim"]]
    _busy["v"] = True; slider.max = n-1; slider.value = min(slider.value, n-1); _busy["v"] = False
    _refresh()

def _on_slider(c):
    if not _busy["v"]: _refresh()

def _on_overlay(c):
    _state["overlay"] = c["new"]; _refresh()

def _on_combo(c):
    s, k = c["new"]
    bf, lbl = _load_clustering(s, k)
    FIELDS[2]["data"], FIELDS[2]["title"] = bf, f"Filtered overturning fraction (σ={s/DS:.4g})"
    FIELDS[3]["data"], FIELDS[3]["title"] = lbl, f"K-means clustering (k={k})"
    _state["sigma"], _state["k"] = s, k
    _refresh()

axis_dd.observe(_on_axis, names="value")
slider.observe(_on_slider, names="value")
overlay.observe(_on_overlay, names="value")
combo_dd.observe(_on_combo, names="value")

display(widgets.VBox([combo_dd, axis_dd, slider, overlay]), out)
_refresh()